# SinLlama 1B — Continual Pre-Training
Runs on Modal GPU workbench. Execute cells top to bottom.

## 1. Install dependencies

In [ ]:
!pip install -q torch transformers datasets peft accelerate sentencepiece scikit-learn huggingface_hub hf_transfer

## 2. HuggingFace login
Paste your token from huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login
login(token="hf_your_token_here")

## 3. Clone your repo

In [ ]:
!git clone https://github.com/isuruij/llama-1B.git
%cd llama-1B/Chinese-LLaMA-Alpaca/scripts/training

## 4. Download the Sinhala dataset (isji/sinhala-corpus)

In [ ]:
import os, shutil
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from huggingface_hub import hf_hub_download

os.makedirs('./sinhala_data', exist_ok=True)

print('Downloading sinhala_corpus.txt from isji/sinhala-corpus ...')
src = hf_hub_download(
    repo_id='isji/sinhala-corpus',
    filename='sinhala_corpus.txt',
    repo_type='dataset',
)
shutil.copy(src, './sinhala_data/sinhala_corpus.txt')
print('Done.')

## 5. Run Continual Pre-Training

In [ ]:
!torchrun \
    --nnodes 1 \
    --nproc_per_node 1 \
    run_clm_pt_with_peft.py \
    --model_name_or_path meta-llama/Llama-3.2-1B \
    --tokenizer_name_or_path polyglots/Extended-Sinhala-LLaMA \
    --dataset_dir ./sinhala_data \
    --data_cache_dir ./data_cache \
    --validation_split_percentage 0.001 \
    --per_device_train_batch_size 16 \
    --per_device_eval_batch_size 16 \
    --do_train \
    --seed 42 \
    --bf16 \
    --num_train_epochs 1 \
    --lr_scheduler_type cosine \
    --learning_rate 2e-4 \
    --warmup_ratio 0.05 \
    --weight_decay 0.01 \
    --logging_strategy steps \
    --logging_steps 10 \
    --logging_first_step True \
    --save_strategy steps \
    --save_steps 200 \
    --save_total_limit 3 \
    --gradient_accumulation_steps 8 \
    --preprocessing_num_workers 8 \
    --block_size 512 \
    --output_dir ./output_sinllama_1b \
    --overwrite_output_dir \
    --ddp_timeout 30000 \
    --lora_rank 8 \
    --lora_alpha 32 \
    --lora_dropout 0.05 \
    --trainable q_proj,v_proj,k_proj,o_proj,gate_proj,down_proj,up_proj \
    --modules_to_save embed_tokens,lm_head \
    --torch_dtype bfloat16

## 6. Push adapter to HuggingFace (isji)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig

adapter_path = './output_sinllama_1b/pt_lora_model'
repo_id = 'isji/sinllama-1b-cpt'

config = PeftConfig.from_pretrained(adapter_path)
model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    torch_dtype=torch.float16,
)
model = PeftModel.from_pretrained(model, adapter_path)
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

print(f'Adapter pushed to https://huggingface.co/{repo_id}')